In [9]:
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import time
import os

In [ ]:
API_KEY = 'CHAVE_API'
youtube = build('youtube', 'v3', developerKey=API_KEY)

arquivo_entrada = 'unique_ids.csv'
arquivo_saida = 'complete_dataset.csv'

In [11]:
try:
    # Pega todos os IDs do arquivo de entrada
    df_entrada = pd.read_csv(arquivo_entrada)
    todos_ids = set(df_entrada['videoId'].tolist())
    print(f"Total de vídeos no arquivo de entrada: {len(todos_ids)}")
except FileNotFoundError:
    print(f"Erro: O arquivo '{arquivo_entrada}' não foi encontrado.")
    exit()

Total de vídeos no arquivo de entrada: 10267


In [12]:
ids_processados = set()
if os.path.exists(arquivo_saida):
    try:
        df_existente = pd.read_csv(arquivo_saida)
        ids_processados = set(df_existente['videoId'].tolist())
        print(f"Encontrados {len(ids_processados)} vídeos já processados.")
    except Exception as e:
        print(f"Aviso ao ler '{arquivo_saida}': {e}. Iniciando do zero.")

# Subtrai para descobrir quem falta processar nesta rodada
ids_pendentes = list(todos_ids - ids_processados)
print(f"Faltam processar: {len(ids_pendentes)} vídeos.")

if len(ids_pendentes) == 0:
    print("Todos os vídeos já foram processados! Saindo...")
    exit()

Faltam processar: 10267 vídeos.


In [13]:
def dividir_em_lotes(lista, tamanho_lote=50):
    return [lista[i:i + tamanho_lote] for i in range(0, len(lista), tamanho_lote)]

lotes = dividir_em_lotes(ids_pendentes)
dados_enriquecidos = []
lotes_processados = 0

print("Iniciando a extração das métricas...")

try:
    for lote in lotes:
        # Transforma a lista de 50 IDs em uma única string
        ids_string = ','.join(lote)
        
        request = youtube.videos().list(
            part="snippet,statistics,contentDetails,topicDetails,status",
            id=ids_string
        )
        response = request.execute()
        
        for item in response.get('items', []):
            snippet = item.get('snippet', {})
            stats = item.get('statistics', {})
            content = item.get('contentDetails', {})
            topics = item.get('topicDetails', {})
            status = item.get('status', {})
            
            topic_categories = topics.get('topicCategories', [])
            topic_names = [url.split('/')[-1] for url in topic_categories] if topic_categories else []
            
            video_info = {
                'videoId': item['id'],
                'title': snippet.get('title', ''),
                'channelTitle': snippet.get('channelTitle', ''),
                'channelId': snippet.get('channelId', ''),
                'publishedAt': snippet.get('publishedAt', ''),
                'description': snippet.get('description', ''),
                'tags': ', '.join(snippet.get('tags', [])),
                'categoryId': snippet.get('categoryId', ''),
                'duration': content.get('duration', ''),
                'caption': content.get('caption', ''),
                'viewCount': stats.get('viewCount', 0),
                'likeCount': stats.get('likeCount', 0),
                'commentCount': stats.get('commentCount', 0),
                'algorithmTopics': ', '.join(topic_names),
                'madeForKids': status.get('madeForKids', False),
                'defaultAudioLanguage': snippet.get('defaultAudioLanguage', '')
            }
            dados_enriquecidos.append(video_info)
            
        lotes_processados += 1
        print(f"Lotes processados: {lotes_processados}/{len(lotes)} | Vídeos extraídos: {len(dados_enriquecidos)}")
        
        if len(dados_enriquecidos) >= 500:
            df_temp = pd.DataFrame(dados_enriquecidos)
            if os.path.exists(arquivo_saida):
                df_temp.to_csv(arquivo_saida, mode='a', header=False, index=False)
            else:
                df_temp.to_csv(arquivo_saida, mode='w', header=True, index=False)
            
            dados_enriquecidos.clear()
            print("   Lote salvo no disco. Memória liberada.")

        time.sleep(0.5)

except HttpError as e:
    print(f"\nErro HTTP da API (Cota excedida?): {e}")
except Exception as e:
    print(f"\nErro inesperado: {e}")

Iniciando a extração das métricas...
Lotes processados: 1/206 | Vídeos extraídos: 50
Lotes processados: 2/206 | Vídeos extraídos: 100
Lotes processados: 3/206 | Vídeos extraídos: 150
Lotes processados: 4/206 | Vídeos extraídos: 200
Lotes processados: 5/206 | Vídeos extraídos: 250
Lotes processados: 6/206 | Vídeos extraídos: 300
Lotes processados: 7/206 | Vídeos extraídos: 350
Lotes processados: 8/206 | Vídeos extraídos: 400
Lotes processados: 9/206 | Vídeos extraídos: 450
Lotes processados: 10/206 | Vídeos extraídos: 500
   Lote salvo no disco. Memória liberada.
Lotes processados: 11/206 | Vídeos extraídos: 50
Lotes processados: 12/206 | Vídeos extraídos: 100
Lotes processados: 13/206 | Vídeos extraídos: 150
Lotes processados: 14/206 | Vídeos extraídos: 200
Lotes processados: 15/206 | Vídeos extraídos: 250
Lotes processados: 16/206 | Vídeos extraídos: 300
Lotes processados: 17/206 | Vídeos extraídos: 350
Lotes processados: 18/206 | Vídeos extraídos: 400
Lotes processados: 19/206 | Víde

In [14]:
if dados_enriquecidos:
    df_temp = pd.DataFrame(dados_enriquecidos)
    
    if os.path.exists(arquivo_saida):
        df_temp.to_csv(arquivo_saida, mode='a', header=False, index=False)
        print(f"\nSUCESSO! Novos vídeos adicionados em '{arquivo_saida}'.")
    else:
        df_temp.to_csv(arquivo_saida, mode='w', header=True, index=False)
        print(f"\nSUCESSO! Arquivo criado em '{arquivo_saida}'.")
else:
    print("\nNenhum dado pendente na memória para salvar no fechamento.")


SUCESSO! Novos vídeos adicionados em 'complete_dataset.csv'.
